## Implement Semantic Search in Redis

### Installing Libraries and Utilites

In [ ]:
%pip install python-dotenv redis==8.0.0 openai==2.38.0 numpy

### Setting up the Environment

In [ ]:
import os
from dotenv import load_dotenv
import numpy as np

load_dotenv()

# loading redis configurations
redis_hostname = os.getenv("REDIS_HOSTNAME")
redis_password = os.getenv("REDIS_PASSWORD")

# loading azure openai configurations
azure_openai_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
azure_openai_api_key = os.getenv("AZURE_OPENAI_API_KEY")
embeddings_model_name = os.getenv("EMBEDDINGS_MODEL_NAME")

### Setting up the Redis Client

In [ ]:
import redis

redis_client = redis.Redis(
    host = redis_hostname,
    port = 10000,
    ssl = True,
    decode_responses = True,
    password = redis_password
)

### Creating the Azure OpenAI Client

In [ ]:
from openai import AzureOpenAI

azure_openai_client = AzureOpenAI(
    api_key=azure_openai_api_key,
    api_version="2024-02-15-preview",
    azure_endpoint=azure_openai_endpoint
)

### Creating the Embedding Generator Function

In [ ]:
def generate_embeddings(client, text):
    
    response = client.embeddings.create(
        input=text,
        model = embeddings_model_name
    )
    
    embeddings=response.model_dump()
    return embeddings['data'][0]['embedding']
    

### Generate an Embedding for User Query

In [ ]:
user_query = """
How do I build a RAG application using Azure AI Search?
"""

query_embedding = generate_embeddings(
    azure_openai_client,
    user_query
)

query_bytes = np.array(
    query_embedding,
    dtype=np.float32
).tobytes()

### Implement Basic KNN Search

In [ ]:
from redis.commands.search.query import Query

query = (
    Query(
        "*=>[KNN 5 @embedding $query_vec AS score]"
    )
    .return_fields(
        "id",
        "title",
        "category",
        "content",
        "score"
    )
    .sort_by("score")
    .dialect(2)
)

results = redis_client.ft(
    "idx:azure-ai-docs"
).search(
    query,
    query_params={
        "query_vec": query_bytes
    }
)

for doc in results.docs:

    print(doc.id)
    print(doc.title)
    print(doc.category)
    print(doc.content)
    print(doc.score)
    print("======================== \n")

### Increase Accuracy with EF_RUNTIME

In [ ]:
from redis.commands.search.query import Query

query = (
    Query(
        "*=>[KNN 5 @embedding $query_vec EF_RUNTIME 100 AS score]"
    )
    .return_fields(
        "id",
        "title",
        "category",
        "content",
        "score"
    )
    .sort_by("score")
    .dialect(2)
)

results = redis_client.ft(
    "idx:azure-ai-docs"
).search(
    query,
    query_params={
        "query_vec": query_bytes
    }
)

for doc in results.docs:

    print(doc.id)
    print(doc.title)
    print(doc.category)
    print(doc.content)
    print(doc.score)
    print("======================== \n")

### Run a Hybrid Search Query (Vector Search with Metadata Filter)

In [ ]:
query = (
    Query(
        "@category:{rag}"
        "=>[KNN 5 @embedding $query_vec AS score]"
    )
    .return_fields(
        "title",
        "category",
        "content",
        "score"
    )
    .sort_by("score")
    .dialect(2)
)

results = redis_client.ft(
    "idx:azure-ai-docs"
).search(
    query,
    query_params={
        "query_vec": query_bytes
    }
)

for doc in results.docs:

    print(doc.id)
    print(doc.title)
    print(doc.category)
    print(doc.content)
    print(doc.score)
    print("======================== \n")